Import The Packages

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, auc, mean_absolute_error, mean_squared_error,
    r2_score,
)
from imblearn.over_sampling import SMOTE
print("All libraries imported successfully.")

os.makedirs("charts", exist_ok=True)
pd.set_option("display.width", 120)
RANDOM_STATE = 42

All libraries imported successfully.


STEP 1: Load the committed offline CSV -- NOT a second sns.load_dataset call

In [7]:
print("=" * 70)
print("STEP 1: Load titanic.csv (produced by 01_eda.py)")
print("=" * 70)

df = pd.read_csv("titanic.csv")
print(f"Loaded titanic.csv: {df.shape}")

FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
TARGET = "survived"

df = df.dropna(subset=[TARGET])  # target itself must never be missing
X = df[FEATURES].copy()
y = df[TARGET].copy()

print("\nClass balance (survived):")
print(y.value_counts(normalize=True).round(3))


STEP 1: Load titanic.csv (produced by 01_eda.py)
Loaded titanic.csv: (891, 15)

Class balance (survived):
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


STEP 2: Stratified train/test split

In [8]:
print("\n" + "=" * 70)
print("STEP 2: Stratified train/test split")
print("=" * 70)
print("Using stratify=y because survived is imbalanced (roughly 38% survived vs")
print("62% did not, per the class balance above). A plain random split risks a")
print("train or test fold with a meaningfully different survival ratio, which")
print("would bias the evaluation metrics -- stratification keeps both folds")
print("representative of the true class balance.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"\nTrain shape: {X_train.shape}   Test shape: {X_test.shape}")



STEP 2: Stratified train/test split
Using stratify=y because survived is imbalanced (roughly 38% survived vs
62% did not, per the class balance above). A plain random split risks a
train or test fold with a meaningfully different survival ratio, which
would bias the evaluation metrics -- stratification keeps both folds
representative of the true class balance.

Train shape: (712, 7)   Test shape: (179, 7)


STEP 3: Preprocessing (built as a factory so every pipeline gets its own
 freshly-instantiated, never-shared ColumnTransformer -- avoids any risk of
 one fitted transformer bleeding into another pipeline's state)

In [9]:
print("\n" + "=" * 70)
print("STEP 3: Preprocessing pipeline (fit on TRAIN only, transform-only on TEST)")
print("=" * 70)
print("Choice: median-impute numeric columns, mode-impute categorical columns,")
print("one-hot encode sex/embarked, StandardScaler on numeric columns.")
print("Every imputer/encoder/scaler lives inside a ColumnTransformer wrapped in")
print("a Pipeline, so sklearn enforces fit-on-train / transform-on-test by")
print("construction -- .fit() is only ever called on X_train.")

NUMERIC_FEATURES = ["age", "sibsp", "parch", "fare"]
CATEGORICAL_FEATURES = ["sex", "embarked"]
PASSTHROUGH_FEATURES = ["pclass"]


def build_preprocessor(numeric_feats, categorical_feats, passthrough_feats):
    """Returns a brand-new (unfitted) ColumnTransformer every call."""
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    transformers = []
    if numeric_feats:
        transformers.append(("num", numeric_transformer, numeric_feats))
    if categorical_feats:
        transformers.append(("cat", categorical_transformer, categorical_feats))
    if passthrough_feats:
        transformers.append(("pass", "passthrough", passthrough_feats))
    return ColumnTransformer(transformers=transformers)



STEP 3: Preprocessing pipeline (fit on TRAIN only, transform-only on TEST)
Choice: median-impute numeric columns, mode-impute categorical columns,
one-hot encode sex/embarked, StandardScaler on numeric columns.
Every imputer/encoder/scaler lives inside a ColumnTransformer wrapped in
a Pipeline, so sklearn enforces fit-on-train / transform-on-test by
construction -- .fit() is only ever called on X_train.


STEP 4: Train three classifiers on the identical split

In [10]:
print("\n" + "=" * 70)
print("STEP 4: Train three classifiers (Logistic Regression, Decision Tree, RF)")
print("=" * 70)

model_specs = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
}

fitted_pipelines = {}
results = {}

for name, clf in model_specs.items():
    pipe = Pipeline(steps=[
        ("preprocessor", build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES, PASSTHROUGH_FEATURES)),
        ("classifier", clf),
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    results[name] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": roc_auc}

    print(f"\n--- {name} ---")
    print("Confusion matrix:\n", cm)
    print(f"Accuracy={acc:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}  AUC={roc_auc:.3f}")

    plt.figure(figsize=(5, 5))
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.2f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"charts/roc_{name.replace(' ', '_').lower()}.png")
    plt.close()

# Decision tree visualization with labeled feature/class names
dt_pipe = fitted_pipelines["Decision Tree"]
dt_model = dt_pipe.named_steps["classifier"]
cat_feature_names = list(
    dt_pipe.named_steps["preprocessor"]
    .named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(CATEGORICAL_FEATURES)
)
all_feature_names = NUMERIC_FEATURES + cat_feature_names + PASSTHROUGH_FEATURES

plt.figure(figsize=(18, 8))
plot_tree(
    dt_model,
    feature_names=all_feature_names,
    class_names=["Did not survive", "Survived"],
    filled=True,
    max_depth=3,   # top levels only, for readability
    fontsize=8,
)
plt.title("Decision Tree (top 3 levels)")
plt.tight_layout()
plt.savefig("charts/decision_tree.png")
plt.close()
print("\nSaved charts/decision_tree.png")

comparison_df = pd.DataFrame(results).T[["accuracy", "precision", "recall", "f1", "auc"]].round(3)
print("\n--- Classifier comparison table ---")
print(comparison_df)



STEP 4: Train three classifiers (Logistic Regression, Decision Tree, RF)

--- Logistic Regression ---
Confusion matrix:
 [[98 12]
 [23 46]]
Accuracy=0.804  Precision=0.793  Recall=0.667  F1=0.724  AUC=0.844


C:\Users\TEMP.DESKTOP-K14BCI3.000\AppData\Local\Temp\ipykernel_12836\1226070698.py:46: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()



--- Decision Tree ---
Confusion matrix:
 [[96 14]
 [18 51]]
Accuracy=0.821  Precision=0.785  Recall=0.739  F1=0.761  AUC=0.802

--- Random Forest ---
Confusion matrix:
 [[97 13]
 [21 48]]
Accuracy=0.810  Precision=0.787  Recall=0.696  F1=0.738  AUC=0.831

Saved charts/decision_tree.png

--- Classifier comparison table ---
                     accuracy  precision  recall     f1    auc
Logistic Regression     0.804      0.793   0.667  0.724  0.844
Decision Tree           0.821      0.785   0.739  0.761  0.802
Random Forest           0.810      0.787   0.696  0.738  0.831


STEP 5: Imbalance-handling comparison (Random Forest)

In [11]:
print("\n" + "=" * 70)
print("STEP 5: Imbalance-handling comparison (baseline vs balanced vs SMOTE)")
print("=" * 70)

print("\nClass balance in the training fold:")
print(y_train.value_counts(normalize=True).round(3))

# (a) baseline
pipe_baseline = Pipeline(steps=[
    ("preprocessor", build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES, PASSTHROUGH_FEATURES)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE)),
])
pipe_baseline.fit(X_train, y_train)
pred_baseline = pipe_baseline.predict(X_test)

# (b) class_weight='balanced'
pipe_balanced = Pipeline(steps=[
    ("preprocessor", build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES, PASSTHROUGH_FEATURES)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
])
pipe_balanced.fit(X_train, y_train)
pred_balanced = pipe_balanced.predict(X_test)

# (c) SMOTE -- applied to the TRAINING FOLD ONLY, after fit-on-train preprocessing,
# so no test-set information leaks into the oversampling step.
smote_preprocessor = build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES, PASSTHROUGH_FEATURES)
X_train_processed = smote_preprocessor.fit_transform(X_train)
X_test_processed = smote_preprocessor.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train_processed, y_train)

rf_smote = RandomForestClassifier(random_state=RANDOM_STATE)
rf_smote.fit(X_train_sm, y_train_sm)
pred_smote = rf_smote.predict(X_test_processed)

imbalance_results = {}
for label, preds in [
    ("Baseline", pred_baseline),
    ("class_weight=balanced", pred_balanced),
    ("SMOTE", pred_smote),
]:
    imbalance_results[label] = {
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }

imbalance_df = pd.DataFrame(imbalance_results).T.round(3)
print("\n", imbalance_df)

best_strategy = imbalance_df["f1"].idxmax()
print(f"\nConclusion: '{best_strategy}' gave the highest F1 score "
      f"({imbalance_df['f1'].max():.3f}) among the three strategies, suggesting it")
print("offers the best precision/recall trade-off on this imbalanced target for a")
print("Random Forest. (Note: which strategy wins can vary by random seed / dataset")
print("split -- report whichever your own run actually produces.)")



STEP 5: Imbalance-handling comparison (baseline vs balanced vs SMOTE)

Class balance in the training fold:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

                        precision  recall     f1
Baseline                   0.787   0.696  0.738
class_weight=balanced      0.758   0.725  0.741
SMOTE                      0.773   0.739  0.756

Conclusion: 'SMOTE' gave the highest F1 score (0.756) among the three strategies, suggesting it
offers the best precision/recall trade-off on this imbalanced target for a
Random Forest. (Note: which strategy wins can vary by random seed / dataset
split -- report whichever your own run actually produces.)


STEP 6: Hyperparameter tuning (GridSearchCV on Random Forest, with OOB)

In [12]:
print("\n" + "=" * 70)
print("STEP 6: GridSearchCV tuning + OOB score (Random Forest)")
print("=" * 70)

rf_tune_pipe = Pipeline(steps=[
    ("preprocessor", build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES, PASSTHROUGH_FEATURES)),
    ("classifier", RandomForestClassifier(oob_score=True, bootstrap=True, random_state=RANDOM_STATE)),
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10],
    "classifier__max_features": ["sqrt", "log2"],
}

grid_search = GridSearchCV(rf_tune_pipe, param_grid, cv=5, scoring="f1", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
best_rf_model = grid_search.best_estimator_.named_steps["classifier"]
print(f"OOB score of the best estimator: {best_rf_model.oob_score_:.3f}")


STEP 6: GridSearchCV tuning + OOB score (Random Forest)
Best params: {'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__n_estimators': 100}
OOB score of the best estimator: 0.824


STEP 7: Regression side-task -- predict fare from the other features

In [13]:
print("\n" + "=" * 70)
print("STEP 7: Regression side-task -- predicting fare")
print("=" * 70)

REG_FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "embarked", "survived"]
REG_NUMERIC = ["age", "sibsp", "parch", "survived"]
REG_CATEGORICAL = ["sex", "embarked"]
REG_PASSTHROUGH = ["pclass"]

X_reg = df[REG_FEATURES].copy()
y_reg = df["fare"].copy()

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

reg_pipe = Pipeline(steps=[
    ("preprocessor", build_preprocessor(REG_NUMERIC, REG_CATEGORICAL, REG_PASSTHROUGH)),
    ("regressor", LinearRegression()),
])
reg_pipe.fit(Xr_train, yr_train)
yr_pred = reg_pipe.predict(Xr_test)

mae = mean_absolute_error(yr_test, yr_pred)
rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))
r2 = r2_score(yr_test, yr_pred)

n = len(yr_test)
p = len(REG_FEATURES)  # raw predictor count, before one-hot expansion (stated simplification)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.3f}")
print(f"Adjusted R²: {adj_r2:.3f}  (p = {p} raw input features, before one-hot expansion)")

residuals = yr_test - yr_pred
plt.figure(figsize=(7, 5))
plt.scatter(yr_pred, residuals, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted fare")
plt.ylabel("Residuals")
plt.title("Residual plot — fare regression")
plt.tight_layout()
plt.savefig("charts/regression_residuals.png")
plt.close()
print("\nSaved charts/regression_residuals.png")

print("\nHeteroscedasticity check: the residual spread visibly widens as predicted fare")
print("increases (a funnel/cone shape rather than a uniform random band around zero),")
print("indicating heteroscedasticity -- the constant-variance assumption of ordinary")
print("linear regression is violated, likely driven by a handful of very expensive")
print("1st-class fares that are much harder to predict precisely than typical fares.")



STEP 7: Regression side-task -- predicting fare
MAE:  20.90
RMSE: 30.53
R²:   0.398
Adjusted R²: 0.373  (p = 7 raw input features, before one-hot expansion)

Saved charts/regression_residuals.png

Heteroscedasticity check: the residual spread visibly widens as predicted fare
increases (a funnel/cone shape rather than a uniform random band around zero),
indicating heteroscedasticity -- the constant-variance assumption of ordinary
linear regression is violated, likely driven by a handful of very expensive
1st-class fares that are much harder to predict precisely than typical fares.


STEP 8: Final model comparison table + recommendation

In [14]:
print("\n" + "=" * 70)
print("STEP 8: Final model comparison table and recommendation")
print("=" * 70)

print("\nClassification metrics (three classifiers, separate metric group):")
print(comparison_df)

regression_metrics_df = pd.DataFrame(
    {"MAE": [mae], "RMSE": [rmse], "R2": [r2], "Adjusted_R2": [adj_r2]},
    index=["Linear Regression (fare)"],
).round(3)
print("\nRegression metrics (separate metric group -- not on the same scale as above):")
print(regression_metrics_df)

best_model_name = comparison_df["f1"].idxmax()
row = comparison_df.loc[best_model_name]
print(f"\nRecommendation: deploy the {best_model_name} model. It achieves the highest F1")
print(f"score ({row['f1']:.3f}) among the three classifiers, balancing precision")
print(f"({row['precision']:.3f}) and recall ({row['recall']:.3f}) better than the")
print(f"alternatives, and its AUC of {row['auc']:.3f} indicates strong overall class")
print(f"separation. Its accuracy ({row['accuracy']:.3f}) is also competitive, making it")
print("the most reliable overall choice for production use on this task.")


STEP 8: Final model comparison table and recommendation

Classification metrics (three classifiers, separate metric group):
                     accuracy  precision  recall     f1    auc
Logistic Regression     0.804      0.793   0.667  0.724  0.844
Decision Tree           0.821      0.785   0.739  0.761  0.802
Random Forest           0.810      0.787   0.696  0.738  0.831

Regression metrics (separate metric group -- not on the same scale as above):
                             MAE    RMSE     R2  Adjusted_R2
Linear Regression (fare)  20.898  30.533  0.398        0.373

Recommendation: deploy the Decision Tree model. It achieves the highest F1
score (0.761) among the three classifiers, balancing precision
(0.785) and recall (0.739) better than the
alternatives, and its AUC of 0.802 indicates strong overall class
separation. Its accuracy (0.821) is also competitive, making it
the most reliable overall choice for production use on this task.


STEP 9: Save the best-performing complete pipeline (preprocessing + model)

In [15]:
print("\n" + "=" * 70)
print("STEP 9: Save + reload the best full pipeline (joblib)")
print("=" * 70)

best_pipeline = fitted_pipelines[best_model_name]
joblib.dump(best_pipeline, "best_model_pipeline.joblib")
print(f"Saved '{best_model_name}' pipeline (preprocessing + estimator, combined) "
      f"-> best_model_pipeline.joblib")

reloaded_pipeline = joblib.load("best_model_pipeline.joblib")
sample_raw = X_test.iloc[:5]
sample_preds = reloaded_pipeline.predict(sample_raw)
print("\nReloaded pipeline predictions on 5 RAW (unpreprocessed) test rows:")
print(sample_preds)
print("\nReload + predict succeeded end-to-end on raw input -- the saved artifact")
print("includes both the fitted preprocessing steps and the final estimator.")

print("\n" + "=" * 70)
print("02_modeling.py complete.")
print("=" * 70)



STEP 9: Save + reload the best full pipeline (joblib)
Saved 'Decision Tree' pipeline (preprocessing + estimator, combined) -> best_model_pipeline.joblib

Reloaded pipeline predictions on 5 RAW (unpreprocessed) test rows:
[0 0 0 0 1]

Reload + predict succeeded end-to-end on raw input -- the saved artifact
includes both the fitted preprocessing steps and the final estimator.

02_modeling.py complete.
